In [1]:
# Cell 1 — Đọc PDF
import sys
sys.path.append('../')
import pdfplumber

pdf_path = '../data/raw/sop/directive_2008_48_EC.pdf'

with pdfplumber.open(pdf_path) as pdf:
    print(f"Số trang: {len(pdf.pages)}")
    print(f"\n--- Trang 1 (xem thử) ---")
    print(pdf.pages[0].extract_text()[:2000])

Số trang: 27

--- Trang 1 (xem thử) ---
L 133/66 EN Official Journal of the European Union 22.5.2008
DIRECTIVES
DIRECTIVE 2008/48/EC OF THE EUROPEAN PARLIAMENTAND OF THE COUNCIL
of 23 April 2008
on credit agreements for consumers and repealing Council Directive 87/102/EEC
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE in the field of credit for natural persons in general and
EUROPEANUNION, consumer credit in particular. An analysis of the national
lawstransposingDirective87/102/EECshowsthatMember
Statesuseavarietyofconsumerprotectionmechanisms,in
Having regard to the Treaty establishing the European Commu- additiontoDirective87/102/EEC,onaccountofdifferences
nity, and in particular Article 95 thereof, in the legal or economic situation at national level.
Having regard to the proposal from the Commission,
(4) The de facto and de jure situation resulting from those
national differences in some cases leads to distortions of
competition among creditors in the Community and
Having regard to

In [2]:
# Cell 3 — Trích xuất toàn bộ text
with pdfplumber.open(pdf_path) as pdf:
    full_text = ''
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            full_text += f"\n--- PAGE {i+1} ---\n{text}"

print(f"Tổng số ký tự: {len(full_text)}")
print(f"\n--- 3000 ký tự đầu ---")
print(full_text[:3000])

Tổng số ký tự: 101262

--- 3000 ký tự đầu ---

--- PAGE 1 ---
L 133/66 EN Official Journal of the European Union 22.5.2008
DIRECTIVES
DIRECTIVE 2008/48/EC OF THE EUROPEAN PARLIAMENTAND OF THE COUNCIL
of 23 April 2008
on credit agreements for consumers and repealing Council Directive 87/102/EEC
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE in the field of credit for natural persons in general and
EUROPEANUNION, consumer credit in particular. An analysis of the national
lawstransposingDirective87/102/EECshowsthatMember
Statesuseavarietyofconsumerprotectionmechanisms,in
Having regard to the Treaty establishing the European Commu- additiontoDirective87/102/EEC,onaccountofdifferences
nity, and in particular Article 95 thereof, in the legal or economic situation at national level.
Having regard to the proposal from the Commission,
(4) The de facto and de jure situation resulting from those
national differences in some cases leads to distortions of
competition among creditors in the Communit

In [3]:
# Cell 4 — Tìm các Article liên quan đến BPI 2017
import re

# Tách theo Article
articles = re.split(r'Article\s+(\d+)', full_text)
print(f"Tổng số Article: {len(articles)//2}")

# In tiêu đề từng Article
for i in range(1, len(articles)-1, 2):
    art_num  = articles[i]
    art_text = articles[i+1][:200].strip().replace('\n', ' ')
    print(f"Article {art_num}: {art_text[:100]}...")

Tổng số Article: 79
Article 95: thereof, in the legal or economic situation at national level. Having regard to the proposal from th...
Article 1: protection of personal data, the right to property, non- Subject matter discrimination, protection o...
Article 2: achieved bytheMember States andcan thereforebe better achieved at Community level, the Community may...
Article 5: a of Decision 1999/468/EC. where the credit has to be repaid within one month; (50) In accordance wi...
Article 10: (1), points (a) to (i), (l) and (r) of...
Article 10: (2), prevailing on the market or free of interest or on other...
Article 10: (4), Articles 11, 13, 16 and Articles 18 to 32 shall terms which are more favourable to the consumer...
Article 4: (1), Arti- (a) such arrangements would be likely to avert the possibility cle 4(2)(a) to (c),...
Article 4: (4), Articles 6 to 9, Arti- of legal proceedings concerning such default; and cle 10(1),...
Article 10: (4),...
Article 10: (5), Articles 12, 15, 17 and 

In [4]:
# Cell 5 — Xem tần suất activity liên quan đến từng Article
import pandas as pd

df = pd.read_parquet('../data/processed/event_log_clean.parquet')

# Nhóm activity theo Article mapping
article_map = {
    'Article 5  (Pre-contractual info)': [
        'O_Create Offer', 'O_Created',
        'O_Sent (mail and online)', 'O_Sent (online only)'
    ],
    'Article 8  (Creditworthiness)': [
        'A_Concept', 'A_Accepted'
    ],
    'Article 10 (Credit agreement)': [
        'A_Complete', 'O_Accepted'
    ],
    'Article 14 (Right of withdrawal)': [
        'O_Returned', 'A_Cancelled'
    ],
    'Article 7  (Incomplete application)': [
        'A_Incomplete', 'W_Call incomplete files'
    ],
}

print("Tần suất activity theo Article:\n")
for article, activities in article_map.items():
    print(f"{article}:")
    for act in activities:
        count = (df['activity'] == act).sum()
        cases = df[df['activity'] == act]['case_id'].nunique()
        print(f"  {act:<40} {count:>8,} event, {cases:>6,} case")
    print()

Tần suất activity theo Article:

Article 5  (Pre-contractual info):
  O_Create Offer                             38,252 event, 28,512 case
  O_Created                                  38,252 event, 28,512 case
  O_Sent (mail and online)                   35,740 event, 28,192 case
  O_Sent (online only)                        1,482 event,  1,332 case

Article 8  (Creditworthiness):
  A_Concept                                  28,512 event, 28,512 case
  A_Accepted                                 28,512 event, 28,512 case

Article 10 (Credit agreement):
  A_Complete                                 28,471 event, 28,471 case
  O_Accepted                                 15,942 event, 15,942 case

Article 14 (Right of withdrawal):
  O_Returned                                 21,194 event, 20,257 case
  A_Cancelled                                 8,901 event,  8,901 case

Article 7  (Incomplete application):
  A_Incomplete                               18,421 event, 13,495 case
  W_Call incom